# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [47]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # TODO: completa a partir de la imagen
        self.start = (0, 0)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        self.terminal_states = {
            (0, 5): 10.0,
            (2, 2): 2.0,
            (3, 5): -10.0,
        }

        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        r, c = state
        if r < 0 or r >= self.height or c < 0 or c >= self.width:
            return False
        if state in self.walls:
            return False
        return True

    def states(self):
        return [(r, c) for r in range(self.height) for c in range(self.width)
                if (r, c) not in self.walls]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def _next_if_move(self, state, action):
        nxt = (state[0] + action[0], state[1] + action[1])
        return nxt if self.is_valid_state(nxt) else state

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Las probabilidades dependen de si 'state' es resbaloso;
        si el movimiento golpea pared/borde, next_state = state.
        Los estados terminales son absorbentes (permanecen en si mismos).
        """
        if self.is_terminal(state):
            return [(state, 1.0)]

        idx = self.actions.index(action)
        left = self.actions[(idx - 1) % 4]
        right = self.actions[(idx + 1) % 4]

        if state in self.slippery_states:
            p_main, p_side = 0.60, 0.20
        else:
            p_main, p_side = 0.90, 0.05

        outcomes = {}
        for a, p in [(action, p_main), (left, p_side), (right, p_side)]:
            ns = self._next_if_move(state, a)
            outcomes[ns] = outcomes.get(ns, 0.0) + p

        return list(outcomes.items())



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [48]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [49]:
def expected_next_value(grid, state, action, V):
    # sum_{s'} T(s,a,s') V(s')
    return sum(p * V[ns] for ns, p in grid.get_transition_probs(state, action))


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    S = grid.states()
    V = {s: 0.0 for s in S}

    for it in range(1, max_iter + 1):
        delta = 0.0
        V_new = V.copy()
        for s in S:
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
                continue
            q_values = [expected_next_value(grid, s, a, V) for a in grid.actions]
            V_new[s] = grid.get_reward(s) + grid.gamma * max(q_values)
            delta = max(delta, abs(V_new[s] - V[s]))
        V = V_new
        if delta < threshold:
            return V, it

    return V, max_iter


def extract_policy(grid, V):
    # pi*(s) = argmax_a sum T(s,a,s') V(s')
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        q_values = [(a, expected_next_value(grid, s, a, V)) for a in grid.actions]
        policy[s] = max(q_values, key=lambda x: x[1])[0]
    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [50]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    S = grid.states()
    V = {s: 0.0 for s in S}

    for it in range(1, max_iter + 1):
        delta = 0.0
        V_new = V.copy()
        for s in S:
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
                continue
            a = policy[s]
            V_new[s] = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, a, V)
            delta = max(delta, abs(V_new[s] - V[s]))
        V = V_new
        if delta < threshold:
            break

    return V


def policy_improvement(grid, V):
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        q_values = [(a, expected_next_value(grid, s, a, V)) for a in grid.actions]
        policy[s] = max(q_values, key=lambda x: x[1])[0]
    return policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # 1. politica inicial arbitraria
    S = grid.states()
    policy = {s: grid.actions[0] for s in S if not grid.is_terminal(s)}

    history = []
    for it in range(1, max_iter + 1):
        # 2. evaluacion
        V = policy_evaluation(grid, policy, threshold)
        # 3. mejora
        new_policy = policy_improvement(grid, V)
        changed = sum(1 for s in policy if policy[s] != new_policy[s])
        history.append(changed)
        policy = new_policy
        # 4. repetir hasta estabilidad
        if changed == 0:
            break

    return policy, V, history



## Parte 4 — Visualización y comparación


In [51]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [52]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 23

Valores:
 -3.041 |  -2.182 |  -1.169 |   WALL   |  +7.802 | +10.000
 -2.309 |   WALL   |  -0.023 |  +2.207 |  +3.838 |  +7.670
 -1.319 |  -0.192 |  +2.000 |  +0.751 |   WALL   |  +4.987
 -1.632 |  -0.616 |  +0.575 |  -1.569 |  -2.873 | -10.000
 -2.612 |  -3.845 |   WALL   |  -2.538 |  -3.394 |  -4.120

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  ↑  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [18, 6, 1, 0]

Valores:
 -3.041 |  -2.182 |  -1.169 |   WALL   |  +7.802 | +10.000
 -2.309 |   WALL   |  -0.023 |  +2.207 |  +3.838 |  +7.670
 -1.319 |  -0.192 |  +2.000 |  +0.751 |   WALL   |  +4.987
 -1.632 |  -0.616 |  +0.575 |  -1.569 |  -2.873 | -10.000
 -2.612 |  -3.845 |   WALL   |  -2.537 |  -3.393 |  -4.120

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  ↑  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?

R: Desde START, el robot busca la entrega +10. La política óptima lo lleva por la ruta hacia la zona de entrega, ya que el valor esperado de esa ruta es mayor que el de ir directamente a la estación de carga +2.


2. ¿Por qué una recompensa menor podría ser óptima?

R: Una recompensa menor puede ser óptima cuando el camino hacia la recompensa mayor es mucho más largo o más riesgoso, y el costo acumulado de esos pasos extra o la probabilidad de terminar en un estado peor por el camino supera la diferencia entre las dos recompensas.

3. ¿En qué estados el piso resbaloso cambia la decisión?

R: El piso resbaloso cambia la decisión en los estados resbalosos y sus vecinos, el robot tiene solo un 60% de probabilidad de moverse en la dirección elegida y un 20% de desviarse hacia cada lado, esto hace que el robot tenga que considerar el mayor riesgo al elegir su acción.

4. ¿Qué papel cumple el costo por paso `-1`?

R: El costo por paso -1 cumple el rol de penalizar la duración del recorrido, haciendo al agente preferir caminos cortos y desincentivando dar vueltas o tomar rutas largas aunque terminen en una recompensa alta.

5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

R: Porque las probabilidades dependen del tipo de piso, en un piso normal  la acción elegida tiene 90% de probabilidad y las desviaciones 5% cada una, en un piso resbaloso la acción elegida tiene 60% y las desviaciones 20% cada una

Osea que (T(s,a,s')) es una función que depende de s (el estado), no es una tabla fija.

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

R: Al reducir el costo por paso, las rutas largas se vuelven menos costosas, la política desde START sigue buscando la entrega +10, pero el valor de la ruta mejora porque cada movimiento tiene una penalización menor.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

R: Al aumentar el riesgo de resbalar, los valores disminuyen cerca de las zonas resbalosas, la política cambia en (2,1) y (3,3) para evitar rutas más riesgosas, pero desde START la decisión no cambia, el robot sigue buscando la entrega +10.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

R: Con mas paciencia las recompensas futuras tienen mayor importancia, los valores de los estados intermedios aumentan bastante por ejemplo en (3,4) pasa de −2.87 a −1.32. La política permanece igual y el robot sigue prefiriendo la entrega +10 sobre la carga +2.

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.

R: La política en START cambia de buscar la entrega +10 a buscar la carga +2 cuando living_reward está en -1.11 cuando el costo supera este numero resulta más conveniente llegar a la recompensa +2 más cercana.
